# Multimodal AI Model Exploration

This notebook explores **different models** (not used in live class) across modalities:

| Modality | Model | Provider |
|----------|-------|----------|
| Text → Text | `llama-3.3-70b-versatile` | Groq |
| Text → Text | `gemini-2.0-flash` | Google |
| Text → Image | `dall-e-3` | OpenAI |
| Image → Text | `gpt-4o-mini` | OpenAI |
| Text → Audio | `tts-1` | OpenAI |
| Audio → Text | `whisper-1` | OpenAI |
| Video → Text | `gemini-2.0-flash` (frames) | Google |

**Setup:** Copy `.env.example` to `.env` in the project root and add your API keys.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Audio, Image as IPImage, Markdown, display
import io
from PIL import Image

from src.services.text import chat_completion
from src.services.image import generate_image, analyze_image
from src.services.audio import generate_speech, transcribe_audio
from src.services.video import describe_video
from src.config import get_keys

keys = get_keys()
print("OpenAI:", keys.has_openai())
print("Google:", keys.has_google())
print("Groq:", keys.has_groq())

## 1. Text → Text (Groq — Llama 3.3 70B)

In [ ]:
prompt = "List three real-world uses of multimodal AI in healthcare."
result, model = chat_completion(prompt, "groq")
display(Markdown(f"**Model:** `{model}`\n\n{result}"))

## 2. Text → Text (Google Gemini 2.0 Flash)

In [ ]:
prompt = "Explain the difference between RAG and fine-tuning in two paragraphs."
result, model = chat_completion(prompt, "google")
display(Markdown(f"**Model:** `{model}`\n\n{result}"))

## 3. Text → Image (OpenAI DALL-E 3)

In [ ]:
prompt = "A minimalist icon of a brain made of circuit lines, blue gradient background"
img_bytes, model = generate_image(prompt, "openai")
print("Model:", model)
display(IPImage(data=img_bytes))

## 4. Image → Text (OpenAI GPT-4o mini Vision)

Uses the image generated above (or place your own `sample.jpg` in `notebooks/`).

In [ ]:
sample_path = ROOT / "notebooks" / "sample.jpg"
if not sample_path.exists():
    sample_path = ROOT / "outputs" / "generated.png"
    sample_path.parent.mkdir(exist_ok=True)
    with open(sample_path, "wb") as f:
        f.write(img_bytes)

with open(sample_path, "rb") as f:
    image_data = f.read()

result, model = analyze_image(image_data, "Describe this image.", "openai")
display(Markdown(f"**Model:** `{model}`\n\n{result}"))
display(IPImage(filename=str(sample_path), width=400))

## 5. Text → Audio (OpenAI TTS)

In [ ]:
text = "Multimodal AI can understand text, images, audio, and video in one system."
audio_bytes, model = generate_speech(text, "openai")
print("Model:", model)
display(Audio(audio_bytes, autoplay=False))

## 6. Audio → Text (OpenAI Whisper)

Save TTS output and transcribe it to verify the round-trip.

In [ ]:
audio_path = ROOT / "outputs" / "tts_sample.mp3"
audio_path.parent.mkdir(exist_ok=True)
audio_path.write_bytes(audio_bytes)

transcript, model = transcribe_audio(audio_bytes, "tts_sample.mp3", "openai")
display(Markdown(f"**Model:** `{model}`\n\n**Transcript:** {transcript}"))

## 7. Video → Text (Gemini + frame sampling)

Place a short `.mp4` at `notebooks/sample.mp4` or skip if unavailable.

In [ ]:
video_path = ROOT / "notebooks" / "sample.mp4"
if video_path.exists():
    video_data = video_path.read_bytes()
    summary, model = describe_video(video_data, "Summarize this video.", "google")
    display(Markdown(f"**Model:** `{model}`\n\n{summary}"))
else:
    print("Add notebooks/sample.mp4 to run video → text demo.")